# Inżynieria cech - PM2.5 z pogodą

Milestone 2, zadanie 2: złączenie oczyszczonego PM2.5 z pogodą Open-Meteo i policzenie cech pochodnych pod model godzinowego PM2.5. Wynik trafia do `data/silver/pm25_pogoda_2023_2025.parquet`.

Jeden wiersz = jedna stacja w jednej godzinie (UTC). Funkcje są w `scripts/silver_features.py`.

## 1. Wczytanie danych z silver

In [1]:
from pathlib import Path

import polars as pl

cwd = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from scripts.pipeline_config import YEARS, PERIOD, START, END, HOURS
BRONZE_DIR = ROOT / 'data' / 'bronze'
SILVER_DIR = ROOT / 'data' / 'silver'

from scripts import silver_features as feat

measurements = pl.read_parquet(SILVER_DIR / f'pomiary_{PERIOD}.parquet')
weather = pl.read_parquet(SILVER_DIR / f'openmeteo_{PERIOD}.parquet')
stations = pl.read_parquet(BRONZE_DIR / 'stacje.parquet')

print('pomiary:', measurements.shape)
print('pogoda: ', weather.shape)

pomiary: (25819774, 9)
pogoda:  (3340608, 12)


## 2. Złączenie PM2.5 z pogodą

Pełna siatka UTC obejmuje sumę stacji i wszystkie lata z konfiguracji. Brak pomiaru, także przez cały rok, zostaje NULL. Dzięki temu shift(k) oznacza dokładnie k godzin wstecz. Kody stacji i jawne uzupełnienia metadanych są opisane w scripts/pipeline_config.py.

In [2]:
base = feat.build_base(measurements, weather, stations)

pm25_rows = measurements.filter((pl.col('wskaznik') == 'PM2.5') & (pl.col('czas_usredniania') == '1g')).height

assert base.height == weather.height
assert base.select(pl.struct('kod_stacji', 'data_czas_utc').is_duplicated().any()).item() is False
assert base.group_by('kod_stacji').len()['len'].unique().to_list() == [HOURS]
assert base['typ_stacji'].null_count() == 0

print('wiersze:', base.height, '| z pomiarem PM2.5:', base['pm25'].is_not_null().sum(), 'z', pm25_rows)

wiersze: 3340608 | z pomiarem PM2.5: 2440845 z 2440928


Siatka pogody obejmuje pełne lata UTC. Pierwsza godzina GIOŚ z 2023 po przesunięciu należy do 2022 i pozostaje poza zakresem analizy. Przełomy lat wewnątrz zakresu nie tracą historii.

## 3. Cechy

| Grupa | Cechy | Uzasadnienie |
|---|---|---|
| kalendarz | `godzina`, `dzien_tygodnia`, `miesiac`, `weekend`, `swieto`, `dzien_wolny` | ruch i ogrzewanie mają rytm dobowy i tygodniowy; liczone w czasie polskim, bo za nim idzie aktywność ludzi |
| kalendarz cykliczny | `godzina_sin/cos`, `miesiac_sin/cos` | 23:00 jest blisko 0:00, a grudzień blisko stycznia |
| historia PM2.5 | `pm25_lag1/2/3/24/168` | stężenie zmienia się powoli; lag 24 h i 168 h łapie rytm doby i tygodnia |
| okna PM2.5 | `pm25_srednia3h/6h/24h`, `pm25_max24h` | wygładzona historia, odporna na pojedyncze braki |
| wiatr | `wiatr_u`, `wiatr_v` | składowe wektora zamiast kąta: 359° i 1° to prawie ten sam kierunek; przy ciszy oba 0 |
| pogoda pochodna | `cisnienie_zmiana3h`, `temperatura_zmiana24h`, `opad_suma6h/24h` | front atmosferyczny przewietrza, opad wymywa pył |
| ogrzewanie | `stopniodni_grzania` = max(0, 15 °C − temperatura) | im zimniej, tym więcej spalania w domach |
| stacja | `typ_stacji`, `typ_obszaru`, `wysokosc_m`, `siatka_szerokosc/dlugosc` | stacja komunikacyjna w mieście ma inny poziom niż tło poza miastem |

Zmienna docelowa to `pm25` w godzinie t. Historia PM2.5 kończy się na t-1, więc model z tymi cechami prognozuje na godzinę do przodu. Pogoda z godziny t to dane rzeczywiste z reanalizy, czyli zakładamy, że pogodę na tę godzinę znamy (w praktyce z prognozy).

Okna liczymy z `min_samples=1`: średnia z dostępnych godzin, NULL dopiero gdy w całym oknie nie ma pomiaru.

In [3]:
features = feat.add_features(base)

print(features.shape)
features.filter(pl.col('kod_stacji') == 'DsWrocAlWisn').select(
    'data_czas_utc', 'godzina', 'pm25', 'pm25_lag1', 'pm25_srednia3h',
    'wind_speed_10m', 'wind_direction_10m', 'wiatr_u', 'wiatr_v', 'stopniodni_grzania',
).slice(30, 6)

(3340608, 40)


data_czas_utc,godzina,pm25,pm25_lag1,pm25_srednia3h,wind_speed_10m,wind_direction_10m,wiatr_u,wiatr_v,stopniodni_grzania
datetime[μs],i8,f64,f64,f64,f64,f64,f64,f64,f64
2023-01-02 06:00:00,7,9.20079,16.2025,16.1048,9.0,143.0,-5.416335,7.18772,5.8
2023-01-02 07:00:00,8,10.1402,9.20079,13.920697,10.4,160.0,-3.557009,9.772803,5.3
2023-01-02 08:00:00,9,15.2648,10.1402,11.84783,13.0,174.0,-1.35887,12.928785,3.9
2023-01-02 09:00:00,10,15.2486,15.2648,11.535263,13.8,189.0,2.158796,13.630099,2.6
2023-01-02 10:00:00,11,17.7634,15.2486,13.5512,7.8,202.0,2.921931,7.232034,1.5
2023-01-02 11:00:00,12,10.7891,17.7634,16.092267,5.5,191.0,1.049449,5.39895,1.4


## 4. Kontrola wyniku

Najważniejsze: żadna cecha nie może widzieć bieżącej wartości PM2.5 (wyciek danych). Sprawdzamy, że lag 1 to wartość z poprzedniej godziny tej samej stacji, a okna kończą się na t-1. Pierwsze godziny każdej stacji nie mogą brać wartości z poprzedniej stacji.

In [4]:
by_station = features.sort('kod_stacji', 'data_czas_utc')
prev = by_station.with_columns(
    pl.col('pm25').shift(1).over('kod_stacji').alias('prev'),
    pl.col('pm25').shift(1).rolling_mean(3, min_samples=1).over('kod_stacji').alias('mean3'),
)

assert prev.filter(pl.col('pm25_lag1').ne_missing(pl.col('prev'))).height == 0
assert prev.filter((pl.col('pm25_srednia3h') - pl.col('mean3')).abs() > 1e-9).height == 0
assert features.filter(pl.col('data_czas_utc') == pl.col('data_czas_utc').min())['pm25_lag1'].null_count() == features['kod_stacji'].n_unique()
assert features.filter(pl.col('wind_speed_10m') == 0).select((pl.col('wiatr_u') == 0) & (pl.col('wiatr_v') == 0)).to_series().all()
assert features['stopniodni_grzania'].min() >= 0

# Ostatnia godzina UTC należy lokalnie do Nowego Roku po ostatnim roku zakresu.
last = features.filter(pl.col('data_czas_utc') == pl.col('data_czas_utc').max())
assert last['swieto'].all() and last['dzien_wolny'].all()

features.null_count().transpose(include_header=True, column_names=['braki']).filter(pl.col('braki') > 0)


column,braki
str,u32
"""wind_direction_10m""",498
"""pm25""",899763
"""pm25_lag1""",899763
"""pm25_lag2""",899854
"""pm25_lag3""",899943
…,…
"""pm25_srednia6h""",884010
"""pm25_srednia24h""",871658
"""pm25_max24h""",871658


Braki w cechach PM2.5 wynikają z luk w pomiarach. Zostawiamy je w silver - wybór, czy usuwać wiersze, czy uzupełniać braki, należy do etapu gold i zależy od modelu.

Szybki podgląd siły cech - korelacja Pearsona z `pm25` (tylko zależności liniowe, więc to wskazówka, nie selekcja):

In [5]:
numeric = [c for c, t in features.schema.items() if t.is_numeric() and c != 'pm25']

(
    features
    .select(pl.corr('pm25', c).alias(c) for c in numeric)
    .transpose(include_header=True, header_name='cecha', column_names=['korelacja'])
    .sort(pl.col('korelacja').abs(), descending=True)
)

cecha,korelacja
str,f64
"""pm25_lag1""",0.9476
"""pm25_srednia3h""",0.90641
"""pm25_lag2""",0.883423
"""pm25_srednia6h""",0.846694
"""pm25_lag3""",0.821116
…,…
"""godzina""",0.023596
"""miesiac_sin""",0.016346
"""dzien_tygodnia""",-0.015362


## 5. Zapis do silver

In [6]:
feat.save_features(features)
print('Zapisano pełny zakres:', PERIOD, 'oraz wycinek 2024 dla benchmarku')
feat.coverage_report(features)

Zapisano pełny zakres: 2023_2025 oraz wycinek 2024 dla benchmarku


kod_stacji,rok,godziny,pomiary_pm25,pokrycie_proc
str,i32,u32,u32,f64
"""DsChojnowMalMOB""",2023,8760,0,0.0
"""DsChojnowMalMOB""",2024,8784,8452,96.22
"""DsChojnowMalMOB""",2025,8760,0,0.0
"""DsDusznSportMOB""",2023,8760,0,0.0
"""DsDusznSportMOB""",2024,8784,0,0.0
…,…,…,…,…
"""ZpSzczAndrze""",2024,8784,8755,99.67
"""ZpSzczAndrze""",2025,8760,8731,99.67
"""ZpSzczPilsud""",2023,8760,8747,99.85


## 6. Horyzont prognozy: 1 h i 24 h

Tabela nie zakłada jednego horyzontu. Zmienna docelowa to zawsze `pm25` w godzinie t, a horyzont wybieramy w gold przez zestaw cech: model może używać tylko tego, co było znane h godzin wcześniej. Oba modele trenujemy bezpośrednio (direct) na tej samej tabeli, a nie przez 24-krotne powtarzanie modelu 1 h (rekurencja kumuluje błąd).

| Grupa cech | 1 h do przodu | 24 h do przodu |
|---|---|---|
| `pm25_lag1`, `pm25_lag2`, `pm25_lag3` | tak | nie - wartości z ostatniej doby |
| `pm25_srednia3h/6h/24h`, `pm25_max24h` | tak | nie - okna kończą się na t-1 |
| `pm25_lag24`, `pm25_lag168` | tak | tak |
| pogoda z godziny t i cechy pogodowe | tak | tak |
| kalendarz i stacja | tak | tak |

Pogodę z godziny t traktujemy w obu wariantach jako znaną, czyli zakładamy prognozę pogody. Ponieważ to dane rzeczywiste z reanalizy, wynik modelu jest górnym oszacowaniem tego, co dałaby prawdziwa prognoza.

Model 1 h jest zdominowany przez `pm25_lag1` (korelacja 0,95 z `pm25`), więc w praktyce przewiduje „tyle co godzinę temu”. Model 24 h musi przewidzieć zmianę stężenia, więc pogoda ma w nim większą wagę. Oba porównujemy z baseline'em persystencji: `pm25_lag1` dla 1 h i `pm25_lag24` dla 24 h.

Możliwe rozszerzenie dla 24 h: średnie PM2.5 z okien kończących się na t-24 (np. t-48...t-25). Obecnie najbliższa historia dostępna przy tym horyzoncie to sam `pm25_lag24`.

## Dalej: analiza gold i niezależny benchmark

Dalsza analiza i etap gold korzystają z `data/silver/pm25_pogoda_2023_2025.parquet` (zachowujemy pełną siatkę UTC; przy zmianie czasu lokalna godzina może się powtarzać). Niezależny eksperyment porównujący układy A/B/C znajduje się w `07_partycjonowanie_benchmark.ipynb` i czyta wycinek `data/silver/pm25_pogoda_2024.parquet` (nie publikuje danych dla gold).
